## Problem 1: Representing SHA-256 Data

#### Representing 32-bit words

Research:
- Section 2 of FIPS 180-4 defines a WORD as a string of 32 bits for SHA-256.
- Section 3.2 defines addition on words as modulo 2^32, so my representation has to wrap around on overflow.
- A Python `int` has no fixed width, so I would need to mask with `0xFFFFFFFF` after every addition or left shift. Which would require alot of extra work.
- The NumPy docs - (link 1) say NumPy integers have a fixed size and overflow when a value doesn't fit, unlike a Python `int`.

Choice: `np.uint32` (link 2) because it is fixed at 32 bits, so results wrap modulo 2^32 without manual masking.

1. https://numpy.org/doc/stable/user/basics.types.html#overflow-errors
2. https://numpy.org/devdocs/reference/arrays.scalars.html#numpy.uint32

In [2]:
## Code work for Representing 32-bit words

import numpy as np  # load NumPy so we can use its fixed-size integer types

# Make a 32-bit word holding the biggest value that fits (all 32 bits are 1)
word = np.uint32(0xFFFFFFFF)
print(hex(word))              # show it in hex: 0xffffffff

# Turn off NumPy's overflow warning, because we want it to wrap
with np.errstate(over="ignore"):
    # Add 1 to the biggest value which results in it not fitting, so it wraps back round to 0
    print(hex(word + np.uint32(1)))   # shows 0x0

# Do the same sum with a normal Python int
# It just gets bigger (33 bits), so it is no longer a valid 32-bit word
print(hex(0xFFFFFFFF + 1))    # shows 0x100000000

0xffffffff
0x0
0x100000000


### Representing sequences of 32-bit words

Research:
- FIPS 180-4, Section 5.2.1 divides each 512-bit block into sixteen 32-bit words.
- Section 6.2.2 specifies a message schedule of 64 words and a hash value composed of 8 words.
- Every sequence required therefore has a fixed length, and every element is a 32-bit word.
- According to the NumPy documentation (link 1), an array stores items of the same type and size, so each element remains a 32-bit word.

Choice: use a NumPy array with `dtype=np.uint32` (link 2), because it has a fixed length and every element is a 32-bit word that wraps in the same way as the individual words described above.

1. https://numpy.org/doc/stable/reference/arrays.ndarray.html
2. https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.uint32

In [3]:
# A block is 16 words, the message schedule is 64 words and the hash value is 8 words
# Start each one as all zeros, with every item a 32-bit word

block    = np.zeros(16, dtype=np.uint32)
schedule = np.zeros(64, dtype=np.uint32)
state    = np.zeros(8,  dtype=np.uint32)

# Check the lengths and the type of the items
print(len(block), len(schedule), len(state))   # shows 16 64 8
print(block.dtype)                             # shows uint32

# Items wrap like single words: the biggest value plus 1 goes back round to 0
words = np.array([0xFFFFFFFF, 0x1, 0x2], dtype=np.uint32)
print([hex(w) for w in words + np.uint32(1)])  # shows ['0x0', '0x2', '0x3']

16 64 8
uint32
['0x0', '0x2', '0x3']


### Representing input messages

Research:

- FIPS 180-4, Section 5.1.1 pads the message according to its length in bits, so the chosen type must make that length easy to obtain.
- The Python documentation (link 1) describes a `bytes` object as an immutable sequence of whole bytes, where each byte is a number from 0 to 255.
- The length in bits is therefore simply `8 * len(msg)`.
- Text must be converted to bytes before hashing. UTF-8 provides a well-defined byte sequence for any character, including non-ASCII ones.
- Python's `hashlib.sha256` accepts `bytes` (link 2), so using the same type allows the implementation to be tested against it later.

Choice: `bytes`, because it is a sequence of whole bytes, its length is easy to obtain, and it matches the input type of `hashlib`.

1. https://docs.python.org/3/library/stdtypes.html#bytes-objects
2. https://docs.python.org/3/library/hashlib.html

In [5]:
# Turn some text into bytes using UTF-8
msg = "abc".encode("utf-8")
print(msg)                  # shows b'abc'

# The length in bytes, and the length in bits that the padding needs
print(len(msg))             # shows 3
print(8 * len(msg))         # shows 24

# Non-ASCII characters take more than one byte
text = "café"
print(len(text))                    # shows 4 bytes
print(len(text.encode("utf-8")))   # shows 5 bytes

b'abc'
3
24
4
5


### Representing 512-bit message blocks

Research:

- FIPS 180-4, Section 2.2.1 defines `M(i)` as message block i, with a size of m bits, and `M(i)_j` as the jth word of that block.
- Section 5.2.1 specifies that each block is parsed into sixteen 32-bit words, so 512 bits = 64 bytes = 16 words.
- Padding operates on whole bytes, so it is convenient to keep the padded message as `bytes`, split it into 64-byte pieces, and only then convert each piece into 16 words.
- The NumPy documentation for `np.frombuffer` (link 1) states that it interprets raw bytes as an array of a chosen type.
- The dtype `">u4"` (link 2) denotes an unsigned 4-byte integer stored big-endian, which is the byte order the standard uses for words.

Choice: a block is 64 `bytes`, converted into an array of 16 `np.uint32` words via `np.frombuffer` with `">u4"`, because this keeps padding simple and provides the compression function with the words it needs.

1. https://numpy.org/doc/stable/reference/generated/numpy.frombuffer.html
2. https://numpy.org/doc/stable/reference/arrays.dtypes.html